# Récupération des données d'événements - Open Agenda

## Objectif
Récupérer les données d'événements à partir de la plateforme Open Agenda pour la ville de **Lille** et la période **2025**, les filtrer et les structurer pour une utilisation dans la base de données vectorielle.

In [1]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from typing import List, Dict
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

OPENDATASOFT_BASE_URL = "https://public.opendatasoft.com/api/records/1.0/search/"
DATASET_ID = "evenements-publics-openagenda"

## 1. Configuration des paramètres de recherche

Définition des critères de filtrage pour Lille et l'année 2025.

In [2]:
CITY = "Lille"
YEAR = 2025

start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 12, 31)

start_timestamp = int(start_date.timestamp())
end_timestamp = int(end_date.timestamp())

print(f"Recherche d'événements pour {CITY}")
print(f"Période: du {start_date.strftime('%d/%m/%Y')} au {end_date.strftime('%d/%m/%Y')}")
print(f"Timestamps: {start_timestamp} - {end_timestamp}")

Recherche d'événements pour Lille
Période: du 01/01/2024 au 31/12/2025
Timestamps: 1704063600 - 1767135600


## 2. Fonction de récupération des événements depuis Open Agenda

In [3]:
def fetch_openagenda_events(city: str, start_ts: int, end_ts: int, max_results: int = 500) -> List[Dict]:
    events = []
    rows = 100
    start = 0
    
    while len(events) < max_results:
        params = {
            'dataset': DATASET_ID,
            'q': f'location_city:"{city}"', 
            'rows': rows,
            'start': start,
            'facet': ['location_city', 'location_department']
        }
        
        try:
            response = requests.get(OPENDATASOFT_BASE_URL, params=params)
            response.raise_for_status()
            
            data = response.json()
            
            if 'records' not in data or len(data['records']) == 0:
                break
            
            for record in data['records']:
                events.append(record['fields'])
            
            total = data.get('nhits', 0)
            print(f"Récupéré {len(events)}/{total} événements...")
            
            if len(events) >= total or len(data['records']) < rows:
                break
            
            start += rows
            
        except requests.exceptions.RequestException as e:
            print(f"Erreur lors de la récupération des événements: {e}")
            break
    
    print(f"\nTotal d'événements récupérés: {len(events)}")
    return events

## 4. Récupération et traitement des données

In [4]:
raw_events = fetch_openagenda_events(CITY, start_timestamp, end_timestamp, max_results=500)

df_events = pd.DataFrame(raw_events)
df_events.head()

Récupéré 100/16366 événements...
Récupéré 200/16366 événements...
Récupéré 300/16366 événements...
Récupéré 400/16366 événements...
Récupéré 500/16366 événements...

Total d'événements récupérés: 500


,age_min,location_name,location_coordinates,accessibility_label_fr,lastdate_end,uid,timings,description_fr,slug,thumbnail,...,location_access_fr,location_phone,links,location_website,location_imagecredits,location_description_fr,location_image,location_links,location_tags,onlineaccesslink
0,4.0,"Au pied de la Déesse, devant le Furet du Nord","[50.636905, 3.063439]",handicap moteur,2023-04-01T14:00:00+00:00,52657513,"[{""begin"": ""2023-04-01T15:00:00+02:00"", ""end"":...",Pour les enfants à partir de 4 ans,poison-davril-poisson-en-ville-7167607,https://cibul.s3.amazonaws.com/ee614cc9de3d40f...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Mission Locale Lille Avenirs,"[50.630211, 3.074747]",NaN,2023-11-14T11:30:00+00:00,27252539,"[{""begin"": ""2023-11-14T10:00:00+01:00"", ""end"":...",Venez échanger sur les lieux pour exercer votr...,ou-exercer-votre-activite-dentrepreneurse,https://cibul.s3.amazonaws.com/709884470dc3445...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Lille,"[50.614891, 3.038953]",NaN,2024-05-21T14:00:00+00:00,35146841,"[{""begin"": ""2024-05-21T14:00:00+02:00"", ""end"":...",Le GRETA sera présent dans nos locaux afin de ...,greta-presentation-sengager-vers-lemploi-7212045,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,AMELIO | Maison de l'Habitat Durable,"[50.625318, 3.050665]",NaN,2023-12-01T15:00:00+00:00,6715413,"[{""begin"": ""2023-12-01T14:00:00+01:00"", ""end"":...",Atelier en 3 séances pour les personnes retrai...,atelier-bien-chez-soi-n3-conseils-et-financeme...,https://cibul.s3.amazonaws.com/09a33bd06c584bf...,...,Ouvert au public,03 59 00 03 59,"[{""link"": ""mailto:maisonhabitatdurable@lilleme...",http://www.maisonhabitatdurable.lillemetropole.fr,Vincent Lecigne/MEL,La Maison de l'Habitat Durable est ouverte à t...,https://cibul.s3.amazonaws.com/location5672108...,https://www.maisonhabitatdurable-lillemetropol...,NaN,NaN
4,NaN,Gare Saint Sauveur,"[50.627319, 3.069793]",NaN,2023-07-02T18:00:00+00:00,70850948,"[{""begin"": ""2023-07-02T18:30:00+02:00"", ""end"":...",Les concerts de BB · La Guinguette du Cours St-So,kawataro-concert,https://cibul.s3.amazonaws.com/81fc82e26dee441...,...,NaN,NaN,"[{""link"": ""https://www.facebook.com/kawataro.m...",NaN,© Laurent Ghesquière,Ouverte en mars 2009 à l'occasion d'Europe XXL...,https://cibul.s3.amazonaws.com/location1070336...,https://www.facebook.com/garesaintsauveur;http...,NaN,NaN



# 5. Nettoyage des données

1. Analyse des valeurs manquantes
2. Suppression des doublons
3. Nettoyage des champs textuels
4. Validation des dates et coordonnées
5. Standardisation des formats

In [5]:

print(f"Dimensions du DataFrame : {df_events.shape}")

missing_data = df_events.isnull().sum()
missing_percentage = (missing_data / len(df_events) * 100).round(2)
missing_df = pd.DataFrame({
    'Colonne': missing_data.index,
    'Valeurs Manquantes': missing_data.values,
    'Pourcentage': missing_percentage.values
}).sort_values('Valeurs Manquantes', ascending=False)

missing_df_filtered = missing_df[missing_df['Valeurs Manquantes'] > 0]
print("Top 15 colonnes avec le plus de valeurs manquantes :")
print(missing_df_filtered.head(15).to_string(index=False))

key_columns = ['title_fr', 'description_fr', 'location_name', 'location_city', 
               'location_address', 'timings', 'age_min', 'age_max']

for col in key_columns:
    if col in df_events.columns:
        missing = df_events[col].isnull().sum()
        missing_pct = (missing / len(df_events) * 100)
        print(f"{col:20s} : {missing:3d} manquantes ({missing_pct:5.1f}%)")

Dimensions du DataFrame : (500, 50)
Top 15 colonnes avec le plus de valeurs manquantes :
                Colonne  Valeurs Manquantes  Pourcentage
       onlineaccesslink                 496         99.2
          location_tags                 474         94.8
                age_min                 413         82.6
                  links                 409         81.8
     location_access_fr                 403         80.6
            keywords_fr                 396         79.2
  location_imagecredits                 386         77.2
                age_max                 382         76.4
         location_image                 359         71.8
          accessibility                 354         70.8
 accessibility_label_fr                 354         70.8
         location_links                 344         68.8
       location_website                 343         68.6
location_description_fr                 343         68.6
         location_phone                 335         67.0

In [6]:

if 'uid' in df_events.columns:
    duplicates_uid = df_events.duplicated(subset=['uid']).sum()
    print(f"Doublons sur UID : {duplicates_uid}")

duplicates_title = df_events.duplicated(subset=['title_fr', 'location_name']).sum()
print(f"Doublons sur (titre + lieu) : {duplicates_title}")

if 'slug' in df_events.columns:
    duplicates_slug = df_events.duplicated(subset=['slug']).sum()
    print(f"Doublons sur slug : {duplicates_slug}")

text_columns = ['title_fr', 'description_fr']
for col in text_columns:
    if col in df_events.columns:
        empty_strings = (df_events[col] == '').sum()
        null_values = df_events[col].isnull().sum()
        very_short = (df_events[col].str.len() < 10).sum() if df_events[col].dtype == 'object' else 0
        print(f"{col}:")
        print(f"  - Valeurs nulles : {null_values}")
        print(f"  - Chaînes vides : {empty_strings}")
        print(f"  - Textes très courts (<10 car.) : {very_short}")
        
        if df_events[col].dtype == 'object':
            lengths = df_events[col].str.len()
            print(f"  - Longueur moyenne : {lengths.mean():.1f} caractères")
            print(f"  - Longueur min/max : {lengths.min():.0f} / {lengths.max():.0f}")

Doublons sur UID : 0
Doublons sur (titre + lieu) : 6
Doublons sur slug : 0
title_fr:
  - Valeurs nulles : 0
  - Chaînes vides : 0
  - Textes très courts (<10 car.) : 26
  - Longueur moyenne : 35.2 caractères
  - Longueur min/max : 4 / 110
description_fr:
  - Valeurs nulles : 0
  - Chaînes vides : 0
  - Textes très courts (<10 car.) : 3
  - Longueur moyenne : 87.9 caractères
  - Longueur min/max : 4 / 200


In [7]:

df_cleaned = df_events.copy()
print(f"Taille initiale : {len(df_cleaned)} événements")

# Suppression des doublons
initial_count = len(df_cleaned)
if 'uid' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop_duplicates(subset=['uid'])
    print(f"Doublons supprimés (UID) : {initial_count - len(df_cleaned)} événements")
else:
    df_cleaned = df_cleaned.drop_duplicates(subset=['title_fr', 'location_name', 'timings'])
    print(f"Doublons supprimés (titre+lieu+dates) : {initial_count - len(df_cleaned)} événements")

# Suppression des événements sans titre ou description
initial_count = len(df_cleaned)
df_cleaned = df_cleaned.dropna(subset=['title_fr', 'description_fr'])
print(f"Événements sans titre/description supprimés : {initial_count - len(df_cleaned)}")

# Nettoyage des champs textuels
def clean_text(text):
    if pd.isna(text):
        return ""
    # Supprimer les espaces multiples
    text = ' '.join(str(text).split())
    # Supprimer les caractères spéciaux en début/fin
    text = text.strip()
    return text

df_cleaned['title_fr'] = df_cleaned['title_fr'].apply(clean_text)
df_cleaned['description_fr'] = df_cleaned['description_fr'].apply(clean_text)
if 'location_name' in df_cleaned.columns:
    df_cleaned['location_name'] = df_cleaned['location_name'].apply(clean_text)


if 'location_address' in df_cleaned.columns:
    df_cleaned['location_address'] = df_cleaned['location_address'].fillna('Adresse non spécifiée')
if 'age_min' in df_cleaned.columns:
    df_cleaned['age_min'] = df_cleaned['age_min'].fillna(0)
if 'age_max' in df_cleaned.columns:
    df_cleaned['age_max'] = df_cleaned['age_max'].fillna(150)


print(f"Taille finale : {len(df_cleaned)} événements")

Taille initiale : 500 événements
Doublons supprimés (UID) : 0 événements
Événements sans titre/description supprimés : 0
Taille finale : 500 événements


# 6 Préparation du Corpus

1. Récupération de 500 événements via l'API OpenDataSoft
2. Stockage dans un DataFrame pandas pour faciliter l'analyse
3. 50 champs disponibles par événement (titre, description, localisation, dates, etc.)

In [8]:

corpus_df = df_events[['title_fr', 'description_fr', 'location_name', 'location_city', 'location_address', "links"]].copy()


print(f"Corpus : {len(corpus_df)} documents structurés")
corpus_df.head()

Corpus : 500 documents structurés


,title_fr,description_fr,location_name,location_city,location_address,links
0,"Poisson d'avril, poisson en ville",Pour les enfants à partir de 4 ans,"Au pied de la Déesse, devant le Furet du Nord",Lille,Grand'Place Lille,NaN
1,Où exercer votre activité d'entrepreneur.se ?,Venez échanger sur les lieux pour exercer votr...,Mission Locale Lille Avenirs,Lille,"5 Boulevard du Maréchal Vaillant, Lille",NaN
2,GRETA - présentation S'Engager Vers l'Emploi,Le GRETA sera présent dans nos locaux afin de ...,Lille,Lille,59000 Lille,NaN
3,Annulé | Atelier « Bien chez soi » n°3 : Conse...,Atelier en 3 séances pour les personnes retrai...,AMELIO | Maison de l'Habitat Durable,Lille,7 bis rue Racine 59000 Lille,"[{""link"": ""mailto:maisonhabitatdurable@lilleme..."
4,Kawatarō · Concert,Les concerts de BB · La Guinguette du Cours St-So,Gare Saint Sauveur,Lille,17 bd Jean Baptiste Lebas,"[{""link"": ""https://www.facebook.com/kawataro.m..."


# 6 : Génération des Chunks

- Découper nos documents en morceaux de taille optimale pour le traitement.

In [9]:
chunks = []
metadata_list = []

for idx, row in corpus_df.iterrows():
    chunk_text = f"""Titre: {row['title_fr']}
Description: {row['description_fr']}
Lieu: {row['location_name']}, {row['location_city']}"""
    
    chunks.append(chunk_text)
    
    metadata_list.append({
        'event_id': idx,
        'title': row['title_fr'],
        'location': row['location_city']
    })


print(chunks[0][:300] + "..." if len(chunks[0]) > 300 else chunks[0])

Titre: Poisson d'avril, poisson en ville
Description: Pour les enfants à partir de 4 ans
Lieu: Au pied de la Déesse, devant le Furet du Nord, Lille


# 7. Production des Embeddings

Transformer nos chunks textuels en vecteurs numériques pour permettre la recherche sémantique.

## Modèle d'embedding :
  - `all-MiniLM-L6-v2` : Modèle rapide et léger (dimensions 384)

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embeddings = embedding_model.encode(chunks, show_progress_bar=True)

print(f"  {embeddings[0][:10]}")

c:\Users\admin\OneDrive - utopios\Bureau\OC_P7_RAG\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 16/16 [00:08<00:00,  1.93it/s]


  [ 0.06358428  0.13288411  0.20136423  0.02511289 -0.0021184  -0.01817126
  0.0421307   0.0877194   0.06997436 -0.17593655]



# 8. Intégration dans un Vector Store

Stocker nos embeddings dans une base de données vectorielle pour permettre des recherches rapides par similarité.

In [11]:
import faiss
import pickle

embeddings_array = embeddings.astype('float32')
dimension = embeddings_array.shape[1]
n_vectors = embeddings_array.shape[0]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings_array)

# Sauvegarde
faiss.write_index(index, "faiss_index.bin")

metadata_store = {
    'chunks': chunks,
    'metadata': metadata_list
}

with open('faiss_metadata.pkl', 'wb') as f:
    pickle.dump(metadata_store, f)


# Test 


In [12]:

# Question de test
query = "Je cherche un concert de musique ce week-end"
print(f"Question : '{query}'")

query_embedding = embedding_model.encode([query])[0]

query_vector = query_embedding.astype('float32').reshape(1, -1)


k = 3 
print(f"Recherche des {k} événements")
distances, indices = index.search(query_vector, k)

print(f"Résultats trouvés :\n")
for i, (idx, distance) in enumerate(zip(indices[0], distances[0]), 1):
    chunk = chunks[idx]
    metadata = metadata_list[idx]
    print(chunk)


Question : 'Je cherche un concert de musique ce week-end'
Recherche des 3 événements
Résultats trouvés :

Titre: [Concert] Molto Morbidi
Description: A partir du 17 juin, la maison Folie Moulins et la Bulle Café vous invitent à découvrir des artistes locaux les vendredis et les dimanches de l'été !
Lieu: maison Folie Moulins, Lille
Titre: Scène ouverte: Fais découvrir ta musique @ Le Musical
Description: L'occasion de tester de nouvelles compos ou de se confronter à un premier public !
Lieu: Le Musical, Lille
Titre: Soirée vidéo: Woodstock @ Le Musical
Description: Tous les mardi ton Musical te propose une soirée au calme posé, pour commencer la semaine.  Les soirées vidéo te permettent chaque semaine de découvrir un artiste/groupe/genre musical différent.
Lieu: Le Musical, Lille
